In [ ]:
# install python #
python3 --version

In [ ]:
#create a project folder#

mkdir Ultrasonic_Project

#go inside the folder#

cd Ultrasonic_Project

In [ ]:
# Create a virtual environment (recommended) #

python3 -m venv venv

In [ ]:
#activate it #

source venv/bin/activate

In [ ]:
# install python packages #

pip install pyvisa pyvisa-py pyusb numpy pandas matplotlib

In [ ]:
# install hombrew and libusb #
brew install libusb

In [ ]:
# confirm if the oscilloscope is detected #
nano usb_test.py

#confirm usb communication #
import usb.core

dev = usb.core.find(idVendor=0x0699, idProduct=0x0408)

if dev is None:
    print("Oscilloscope not found.")
else:
    print("Oscilloscope found!")
    print("Vendor ID:", hex(dev.idVendor))
    print("Product ID:", hex(dev.idProduct))

#save#
Ctrl + O
Enter
Ctrl + X

#Run#
python usb_test.py

In [ ]:
# Test PyVISA connection #
nano visa_test.py


# paste #

import pyvisa

RESOURCE = "USB0::0x0699::0x0408::C048906::INSTR"

rm = pyvisa.ResourceManager("@py")

print("Using VISA backend:")
print(rm)

print("\nTrying to connect to:")
print(RESOURCE)

scope = rm.open_resource(RESOURCE)

scope.timeout = 10000
scope.write_termination = "\n"
scope.read_termination = "\n"

print("\nOscilloscope identity:")
print(scope.query("*IDN?"))

scope.close()

#save#
Ctrl + O
Enter
Ctrl + X

In [ ]:
# save and run #

python visa_test.py

In [ ]:
# Oscilloscope preparation and Waveform extraction #

# Before running the extraction code, set up the oscilloscope properly.

# On the Tektronix MDO3014:#

#Connect your ultrasonic receiver signal to CH1.
#Turn CH1 ON.
#Adjust the vertical scale until the waveform is visible.
#Adjust the horizontal time scale so the full ultrasonic arrival is visible.
#Set your trigger so the waveform is stable.
#Press Single if you want to capture one waveform.
#Leave the captured waveform displayed on the screen.

#The Python script will download the waveform that is currently available from the oscilloscope.


# Create the waveform extraction script #
#Run#

nano extract_waveform_mdo3014.py

# paste #

import pyvisa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

# ============================================================
# User settings
# ============================================================

RESOURCE = "USB0::0x0699::0x0408::C048906::INSTR"

CHANNEL = "CH1"

START_POINT = 1
STOP_POINT = 10000

OUTPUT_FOLDER = Path("waveform_data")
OUTPUT_FOLDER.mkdir(exist_ok=True)


#for a repetitive experiment#

RESOURCE = "USB0::0x0699::0x0408::C048906::INSTR"

CHANNEL = "CH1"

SAMPLE_ID = "Sample_02"
CORE_TYPE = "Synthetic_Powder_Core"
TRANSDUCER_FREQUENCY_MHZ = 1.0

START_POINT = 1
STOP_POINT = 10000

OUTPUT_FOLDER = Path("waveform_data")
OUTPUT_FOLDER.mkdir(exist_ok=True)


# OR - This for a repeptitive experiment where it asks this question before running #

RESOURCE = "USB0::0x0699::0x0408::C048906::INSTR"

CHANNEL = "CH1"

# Ask for experiment information every time you run the code
SAMPLE_ID = input("Enter sample ID, for example Sample_02: ")
CORE_TYPE = input("Enter core type, for example Synthetic_Powder_Core: ")
TRANSDUCER_FREQUENCY_MHZ = input("Enter transducer frequency in MHz, for example 1.0: ")

START_POINT = 1
STOP_POINT = 10000

OUTPUT_FOLDER = Path("waveform_data")
OUTPUT_FOLDER.mkdir(exist_ok=True)



# ============================================================
# Connect to oscilloscope
# ============================================================

rm = pyvisa.ResourceManager("@py")

scope = rm.open_resource(RESOURCE)

scope.timeout = 20000
scope.chunk_size = 102400

scope.write_termination = "\n"
scope.read_termination = "\n"

print("Connected to oscilloscope:")
print(scope.query("*IDN?"))

# ============================================================
# Configure waveform transfer
# ============================================================

scope.write("HEADER OFF")

scope.write(f"DATA:SOURCE {CHANNEL}")
scope.write(f"DATA:START {START_POINT}")
scope.write(f"DATA:STOP {STOP_POINT}")

# Efficient binary transfer
# RIBinary = signed integer binary data
# WIDTH 2 = 2 bytes per data point
scope.write("DATA:ENCdg RIBinary")
scope.write("DATA:WIDTH 2")

# ============================================================
# Read waveform scaling information
# ============================================================

x_increment = float(scope.query("WFMOutpre:XINcr?"))
x_zero = float(scope.query("WFMOutpre:XZEro?"))

try:
    point_offset = float(scope.query("WFMOutpre:PT_Off?"))
except Exception:
    point_offset = 0.0

y_multiplier = float(scope.query("WFMOutpre:YMUlt?"))
y_zero = float(scope.query("WFMOutpre:YZEro?"))
y_offset = float(scope.query("WFMOutpre:YOFf?"))

x_unit = scope.query("WFMOutpre:XUNit?").strip()
y_unit = scope.query("WFMOutpre:YUNit?").strip()

print("\nWaveform scaling:")
print("Time increment:", x_increment, x_unit)
print("Y multiplier:", y_multiplier)
print("Y zero:", y_zero)
print("Y offset:", y_offset)
print("Voltage unit:", y_unit)

# ============================================================
# Download waveform data
# ============================================================

print("\nDownloading waveform...")

adc_values = scope.query_binary_values(
    "CURVE?",
    datatype="h",
    is_big_endian=True,
    container=np.array
)

print("Number of points downloaded:", len(adc_values))

# ============================================================
# Convert oscilloscope data to real time and voltage
# ============================================================

point_numbers = np.arange(len(adc_values))

time_seconds = x_zero + (point_numbers - point_offset) * x_increment

voltage_volts = (adc_values - y_offset) * y_multiplier + y_zero

# ============================================================
# Save to CSV
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_filename = OUTPUT_FOLDER / f"MDO3014_{CHANNEL}_waveform_{timestamp}.csv"
png_filename = OUTPUT_FOLDER / f"MDO3014_{CHANNEL}_waveform_{timestamp}.png"

df = pd.DataFrame({
    "Point": point_numbers,
    "Time_seconds": time_seconds,
    "Voltage_volts": voltage_volts
})

df.to_csv(csv_filename, index=False)

print("\nCSV file saved:")
print(csv_filename)


#for a repeptitive experiment, replace these above#

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_filename = OUTPUT_FOLDER / f"{SAMPLE_ID}_{CORE_TYPE}_{TRANSDUCER_FREQUENCY_MHZ}MHz_{CHANNEL}_{timestamp}.csv"
png_filename = OUTPUT_FOLDER / f"{SAMPLE_ID}_{CORE_TYPE}_{TRANSDUCER_FREQUENCY_MHZ}MHz_{CHANNEL}_{timestamp}.png"

df = pd.DataFrame({
    "Sample_ID": SAMPLE_ID,
    "Core_Type": CORE_TYPE,
    "Transducer_Frequency_MHz": TRANSDUCER_FREQUENCY_MHZ,
    "Channel": CHANNEL,
    "Point": point_numbers,
    "Time_seconds": time_seconds,
    "Voltage_volts": voltage_volts
})

df.to_csv(csv_filename, index=False)

print("\nCSV file saved:")
print(csv_filename)

# ============================================================
# Plot waveform
# ============================================================

plt.figure()
plt.plot(time_seconds, voltage_volts)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title(f"Ultrasonic Waveform from {CHANNEL}")
plt.grid(True)
plt.tight_layout()
plt.savefig(png_filename, dpi=300)
plt.show()

print("\nPlot saved:")
print(png_filename)

# ============================================================
# Close oscilloscope connection
# ============================================================

scope.close()

print("\nDone.")

# SAVE #

Ctrl + O
Enter
Ctrl + X



In [ ]:
# Run the waveform extraction code #

python extract_waveform_mdo3014.py